# Three experiments in low-rank fine-tuning

What can a small update teach an existing translation model? We tried LoRA on a multilingual encoder–decoder, isolated the effect of changing a small slice of the data on a decoder-only model, then trained that model on a full official corpus. This notebook walks through the experiments in that order.

**Run all cells on a Python 3.9+ CPU kernel.** Small numerical examples demonstrate a low-rank update, the training objective and grouping variable-length messages. The results tables use measurements saved from the original runs. All inputs are embedded. Full GPU training and model download are outside this walkthrough.

## 1. NLLB — adapt attention while keeping the base weights

We used `facebook/nllb-200-distilled-600M` at revision `f8d333a098d19b4fd9a8b18f94170487ad3f821d`. The encoder received the Slavic source, identified by its language tag. The decoder learned to predict the English reference. Polish examples came from JRC-Acquis and Europarl, Croatian from DGT Translation Memory, and Russian from the UN Parallel Corpus. We shuffled 100,000 pairs per language and completed 9,375 updates, each covering 32 pairs. Ukrainian contributed zero training pairs.

LoRA targeted the query and value projections. Rank was 8, alpha was 16, dropout was 0.05 and the initial learning rate was 0.0001. For an input $x$, the projection computes $Wx + (\alpha/r)B(Ax)$. The optimiser can change the second term while the weight matrix $W$ stays fixed.

The cell below takes one gradient step on a miniature two-class problem. Its projection feeds the output logits directly so the entire gradient fits on the page. A real attention layer passes the updated query or value through the rest of the model. The example uses rank one and starts $B$ at zero; at that first step the gradient of $A$ is zero.

In [1]:
from math import ceil, exp, isclose, log


def dot(left, right):
    return sum(a * b for a, b in zip(left, right))


def probabilities(logits):
    largest = max(logits)
    weights = [exp(value - largest) for value in logits]
    total = sum(weights)
    return [weight / total for weight in weights]


frozen_weight = ((0.3, 0.2, -0.1), (-0.4, 0.1, 0.5))
example = (1.0, -1.0, 0.5)
a_row = (0.2, -0.1, 0.1)
b_column = (0.0, 0.0)
scale = 2.0  # alpha / rank
feature = dot(a_row, example)
base_logits = tuple(dot(row, example) for row in frozen_weight)


def adapted_probabilities(column):
    logits = [z + scale * b * feature for z, b in zip(base_logits, column)]
    return probabilities(logits)


correct_class = 1
before = adapted_probabilities(b_column)
# For softmax cross-entropy, d(loss)/d(logit) = probability - one_hot.
grad_b = [
    (p - int(k == correct_class)) * scale * feature
    for k, p in enumerate(before)
]
learning_rate = 0.5  # Chosen for this small example.
updated_b = tuple(b - learning_rate * g for b, g in zip(b_column, grad_b))
after = adapted_probabilities(updated_b)
assert before == probabilities(base_logits) and after[correct_class] > before[1]
print(f"Probability of target class: {before[1]:.4f} → {after[1]:.4f}")
print(f"Cross-entropy: {-log(before[1]):.4f} → {-log(after[1]):.4f}")

Probability of target class: 0.4256 → 0.4954
Cross-entropy: 0.8544 → 0.7025


## 2. Gemma — hold the training budget fixed and change the examples

Both arms started from Gemma 3 1B-IT, Kaggle release `google/gemma-3/transformers/gemma-3-1b-it/1` (model version ID 282742), with identical initial adapters. On a Tesla T4, each arm completed 48 optimiser updates on 384 pairs. Both used single-example microbatches, eight-way gradient accumulation, seed 42, rank-8 query/value LoRA, alpha 16, zero LoRA dropout and AdamW at 0.0002.

The official arm saw 384 official pairs. The mixed arm saw 312 official pairs plus 72 bilingual-reviewed synthetic everyday pairs from batch B01 (24 per training language). The regression set comprised **72 different messages**, with 18 each in Polish, Croatian, Russian and Ukrainian. The table computes two within-experiment changes in mean WMT22 COMET-DA. The NLLB columns compare the first experiment’s adapter to its own base. The Gemma columns compare its two matched data mixtures after the same number of steps.

In [2]:
# Language, NLLB base / round 1, Gemma official / mixed (48 steps).
everyday_comet = [
    ("pl", 0.6803693, 0.719965, 0.676041, 0.6884526),
    ("hr", 0.792665, 0.8341195, 0.7487137, 0.7483602),
    ("ru", 0.7412337, 0.7633444, 0.7692541, 0.7629184),
    ("uk", 0.6898536, 0.7625005, 0.7721158, 0.7749314),
]

print("Lang    NLLB base → round 1  (change)     Gemma official → mixed  (change)")
for lang, base, round_one, official, mixed in everyday_comet:
    print(f"{lang:4}    {base:.4f} → {round_one:.4f}  ({round_one-base:+.4f})"
          f"          {official:.4f} → {mixed:.4f}  ({mixed-official:+.4f})")

Lang    NLLB base → round 1  (change)     Gemma official → mixed  (change)
pl      0.6804 → 0.7200  (+0.0396)          0.6760 → 0.6885  (+0.0124)
hr      0.7927 → 0.8341  (+0.0415)          0.7487 → 0.7484  (-0.0004)
ru      0.7412 → 0.7633  (+0.0221)          0.7693 → 0.7629  (-0.0063)
uk      0.6899 → 0.7625  (+0.0726)          0.7721 → 0.7749  (+0.0028)


NLLB’s mean COMET rose for each language on this small set. The matched Gemma changes were positive for Polish and Ukrainian, slightly negative for Croatian and negative for Russian. The paired 95% resampling intervals for all four mixed-minus-official differences included zero. These 48-step arms used one seed. The comparison examines the data replacement at that budget.

## 3. Gemma — scale to one full official epoch

We restarted from the same Kaggle base and trained on 300,000 official pairs using a single NVIDIA L4. This data revision replaced 575 malformed Croatian alignments and one malformed Russian alignment from the first pilot. The run used 745,472 trainable parameters, the same rank, alpha and target projections as the short arms, and a 32-pair effective batch. It saved reloadable adapters after 10,000, 50,000 and 300,000 examples. Each checkpoint boundary finished an optimiser update.

### Put the loss on the English answer

Gemma sees a user instruction and source message followed by the English answer in its native chat template. At position $t$, its logits predict token $t+1$. We supervise the answer tokens and their end-of-turn marker. Prompt and padding positions have label `-100`. The full trainer applies the vocabulary projection only to hidden states with a supervised next token. It divides the summed microbatch losses by the total number of supervised tokens in the optimiser update.

These two invented sequences have different answer lengths. In the example we supply the probability assigned to each true next token so the shifted labels and loss calculation can be inspected without a Gemma checkpoint. The integer IDs are illustrative.

In [3]:
ignore = -100
end_of_turn = 106
# The first three IDs represent the prompt. Zeros on the right are padding.
microbatches = [
    ([1, 20, 30, 51, end_of_turn, 0], 3, 5, [0.3, 0.4, 0.8, 0.9, 0.1]),
    ([1, 21, 30, 62, 85, end_of_turn], 3, 6, [0.4, 0.3, 0.2, 0.3, 0.4]),
]

loss_sums = []
token_counts = []
for ids, prefix_length, unpadded_length, true_next_probs in microbatches:
    labels = [ignore] * len(ids)
    labels[prefix_length:unpadded_length] = ids[prefix_length:unpadded_length]
    # labels[1:] aligns one label with each predictive logit row.
    losses = [
        -log(p) for label, p in zip(labels[1:], true_next_probs)
        if label != ignore
    ]
    assert labels[unpadded_length - 1] == end_of_turn
    loss_sums.append(sum(losses))
    token_counts.append(len(losses))
    print(f"Shifted labels {labels[1:]}, supervised tokens {len(losses)}")

update_loss = sum(loss_sums) / sum(token_counts)
mean_of_means = sum(s / n for s, n in zip(loss_sums, token_counts)) / 2
direct_loss = sum(-log(p) for p in [0.8, 0.9, 0.2, 0.3, 0.4]) / 5
assert token_counts == [2, 3] and isclose(update_loss, direct_loss)
print(f"Token-weighted update loss: {update_loss:.4f}")
print(f"Equal-weight mean of microbatch means: {mean_of_means:.4f}")

Shifted labels [-100, -100, 51, 106, -100], supervised tokens 2
Shifted labels [-100, -100, 62, 85, 106], supervised tokens 3
Token-weighted update loss: 0.8116
Equal-weight mean of microbatch means: 0.7037


On the L4, the optimised target-only loss was 3.41570568 on a three-language padded batch. The native Hugging Face loss on that same batch was 3.41570520. The cosine similarity of the checked LoRA gradient tensor was 0.999915.

### Bound the physical batch while retaining long records

The full run shuffled the data, sorted by length inside windows of up to 1,024 examples and shuffled full groups of 32 within each window. Within a group it processed up to eight examples at once, limited by the longest sequence times the number of examples (1,536 tokens before rounding the padding width). A long singleton was retained in full. The cell applies the same capacity rule to a few already sorted lengths. They are multiples of eight, so padding-width rounding does not affect this illustration.

In [4]:
def split_microbatches(lengths, max_pairs=8, token_budget=1536):
    group = []
    for length in lengths:
        candidate = group + [length]
        padded_tokens = len(candidate) * max(candidate)
        if group and (len(candidate) > max_pairs or padded_tokens > token_budget):
            yield group
            group = []
        group.append(length)
    if group:
        yield group


sample_lengths = [48, 56, 64, 72, 80, 88, 96, 104, 112, 512, 640, 2352]
groups = list(split_microbatches(sample_lengths))
for group in groups:
    print(f"Lengths {group}, padded positions {len(group) * max(group)}")
assert [n for group in groups for n in group] == sample_lengths
assert all(len(g) == 1 or len(g) * max(g) <= 1536 for g in groups)

Lengths [48, 56, 64, 72, 80, 88, 96, 104], padded positions 832
Lengths [112, 512], padded positions 1024
Lengths [640], padded positions 640
Lengths [2352], padded positions 2352


The speed measurement used a separate, controlled benchmark on 32 identical length-sorted examples. Moving from single-example microbatches to up to eight raised forward/backward throughput from 5.48 to 32.17 pairs/s (5.88× using the unrounded timings). Peak allocated GPU memory rose from 2.66 to 5.65 GiB. Those timings exclude the optimiser update. The full training timer included the optimiser and transfers.

### Verify the milestone accounting

This last table combines the recorded losses on 48 fixed official validation messages with the number of updates implied by the prespecified checkpoint boundaries. Lower teacher-forced loss means that the model assigned more probability to reference tokens given their preceding context. The separate translation evaluation measures generated output on official text, everyday messages and Ukrainian transfer sets.

In [5]:
# Examples seen and recorded validation loss.
milestone_losses = [
    (0, 3.9534896620675557),
    (10_000, 1.2153047929199605),
    (50_000, 1.0963876771132781),
    (300_000, 0.9503143576758897),
]

effective_batch = 32
previous_examples = 0
steps = 0
print("Examples    Updates    Validation loss")
for examples_seen, loss in milestone_losses:
    steps += ceil((examples_seen - previous_examples) / effective_batch)
    print(f"{examples_seen:>8,}    {steps:>7,}    {loss:.4f}")
    previous_examples = examples_seen

training_seconds = 7889.30262995509
assert steps == 9_376 and previous_examples == 300_000
print(f"Average full training throughput: {previous_examples/training_seconds:.2f} pairs/s")

Examples    Updates    Validation loss
       0          0    3.9535
  10,000        313    1.2153
  50,000      1,563    1.0964
 300,000      9,376    0.9503
Average full training throughput: 38.03 pairs/s


The full run processed 29,955,548 chat tokens, including 9,670,521 supervised English tokens and end-of-turn markers, in 2 h 11 min 29 s of timed training. Official-test translation scores improved in all three training languages. Everyday COMET rose only for Croatian at the final checkpoint. See [the runnable translation evaluation](./evaluate_translation.ipynb) for the per-language measurements and a decision-level example.

The project was developed during the [Google DeepMind AI Research Foundations capstone](https://www.skills.google/paths/3135/course_templates/1557). This walkthrough uses the project’s own experiment records and newly written numerical examples.